# HBOT B8 · Colab 免费 T4 训练 notebook

**用法**：按顺序运行；每次 Colab 重连后从 Cell 1 重跑到 Cell 5，再运行 Cell 7/9 的 Resume。

**前置（必须先在私有 Google Drive 准备）**
- `/content/drive/MyDrive/hbot/hbot-colab-data.tgz` — 由 `deployment/pack_colab_data.sh` 在 ModelScope 实例上生成，含 prepared/冻结控制器/assets/compiled/b1-prior/model_3000
- `/content/drive/MyDrive/hbot/b8-code.zip` — B8 代码（state-conditioned prior、NFSP average/reservoir、export --policy average）。**目前尚未生成；没有它时 notebook 只跑 B7 SL。**
- InterHuman/SMPL 数据只放私有 Drive 或加密包，不要公开分享。

**T4 限制**：16GB 显存；SL/RL 约 3.4GB 可跑；**不要跑 prepare**（峰值约 17GB）。T4 2 vCPU 比 A10 慢；NFSP 按 200 updates/块、每块 1800s 分块跑，checkpoint 每 300s 写 Drive，断了重连续跑。

**防闲置/断连**：训练放 tmux + nohup + nice；Cell 8 的 JS 每分钟模拟点击 Connect；每块 `--max-runtime-seconds 1800` + `--resume`；日志和 results 定时 rsync 到 Drive。


In [ ]:
#@title 1. GPU + Drive + 路径配置
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/hbot'
ROOT = '/content/HoloMotion'
DATA_TAR = f'{DRIVE}/hbot-colab-data.tgz'
CODE_ZIP = f'{DRIVE}/b8-code.zip'
VENV_TAR = f'{DRIVE}/venv-py311.tar.zst'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)
os.makedirs(f'{DRIVE}/logs', exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /content | tail -1
!free -g | head -2
print('DATA_TAR exists:', os.path.isfile(DATA_TAR))
print('CODE_ZIP exists:', os.path.isfile(CODE_ZIP))
print('VENV_TAR exists:', os.path.isfile(VENV_TAR))


In [ ]:
#@title 2. 克隆仓库 + 从 Drive 解包数据/控制器/资产 + 覆盖 B8 代码
!apt-get -qq update && apt-get -qq install -y tmux rsync zstd unzip pv >/dev/null 2>&1
!GIT_LFS_SKIP_SMUDGE=1 git clone --filter=blob:none https://github.com/HorizonRobotics/HoloMotion.git /content/HoloMotion 2>/dev/null || true


In [ ]:
%%bash
set -e
ROOT=/content/HoloMotion
DRIVE=/content/drive/MyDrive/hbot
mkdir -p "$ROOT/models" "$ROOT/results" "$ROOT/deployment" "$ROOT/docs" \
         "$ROOT/holoboxing/data/processed" "$ROOT/holoboxing/evaluation/runtime"
if [ ! -f "$DRIVE/hbot-colab-data.tgz" ]; then
  echo "MISSING $DRIVE/hbot-colab-data.tgz — run deployment/pack_colab_data.sh on ModelScope first"
  exit 1
fi
rm -rf /tmp/hbot_data && mkdir -p /tmp/hbot_data
tar xzf "$DRIVE/hbot-colab-data.tgz" -C /tmp/hbot_data
rsync -a /tmp/hbot_data/HoloMotion/holoboxing/data/processed/b7-boxing "$ROOT/holoboxing/data/processed/"
rsync -a /tmp/hbot_data/HoloMotion/models/holomotion-v2 "$ROOT/models/"
rsync -a /tmp/hbot_data/HoloMotion/holoboxing/assets "$ROOT/holoboxing/"
rsync -a /tmp/hbot_data/HoloMotion/holoboxing/evaluation/runtime/compiled "$ROOT/holoboxing/evaluation/runtime/"
rsync -a /tmp/hbot_data/HoloMotion/results/b1-prior "$ROOT/results/"
cp /tmp/hbot_data/HoloMotion/results/boxing-b1-sl.onnx "$ROOT/results/" 2>/dev/null || true
cp /tmp/hbot_data/HoloMotion/deployment/fall_attrib.py "$ROOT/deployment/" 2>/dev/null || true
rsync -a /tmp/hbot_data/HoloMotion/docs/b8 "$ROOT/docs/" 2>/dev/null || true
cp /tmp/hbot_data/model_3000.pt "$ROOT/model_3000.pt" 2>/dev/null || true
if [ -f "$DRIVE/b8-code.zip" ]; then
  unzip -o "$DRIVE/b8-code.zip" -d "$ROOT"
  echo "B8 code overlaid"
else
  echo "WARN: b8-code.zip missing; B7 SL will run, B8 cells will stop"
fi
echo "DATA_OK"
find "$ROOT/holoboxing/data/processed/b7-boxing/prepared" -maxdepth 1 -name 'clip-*.npz' | wc -l


In [ ]:
#@title 3. 安装/恢复 Python 3.11 训练环境（venv 缓存到 Drive）
!pip install -q uv


In [ ]:
%%bash
set -e
ROOT=/content/HoloMotion
DRIVE=/content/drive/MyDrive/hbot
if [ -f "$DRIVE/venv-py311.tar.zst" ]; then
  echo "restore venv from Drive"
  tar --zstd -xf "$DRIVE/venv-py311.tar.zst" -C "$ROOT"
else
  echo "build venv (first run, ~10-15 min)"
  uv python install 3.11
  uv venv --python 3.11 "$ROOT/.venv-holoboxing"
  source "$ROOT/.venv-holoboxing/bin/activate"
  pip install -q torch==2.9.1 --index-url https://download.pytorch.org/whl/cu128
  pip install -q -r "$ROOT/holoboxing/training/requirements.txt"
  echo "save venv cache to Drive"
  tar --zstd -cf "$DRIVE/venv-py311.tar.zst" -C "$ROOT" .venv-holoboxing
fi
"$ROOT/.venv-holoboxing/bin/python" -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"


### 4. 环境变量 + smoke

In [ ]:
%%writefile /content/env-colab.sh
export HOLOBOXING_ASSETS=/content/HoloMotion/holoboxing/assets
export HOLOBOXING_LOWER=/content/HoloMotion/models/holomotion-v2
export HOLOBOXING_DATA=/content/HoloMotion/holoboxing/data/processed/b7-boxing/prepared
export PIP_CACHE_DIR=/content/.cache/pip
export ROBOSTRIKER_CHECKPOINT=/content/HoloMotion/model_3000.pt


In [ ]:
%%bash
set -e
ROOT=/content/HoloMotion
source /content/env-colab.sh
"$ROOT/.venv-holoboxing/bin/python" -m holoboxing assets --assets "$HOLOBOXING_ASSETS"
"$ROOT/.venv-holoboxing/bin/python" - <<'PY'
from holoboxing.training.supervised.data import Records
p = "/content/HoloMotion/holoboxing/data/processed/b7-boxing/prepared/manifest.json"
r = Records(p)
v = Records(p, split="validation", allow_empty=True)
print("train_records", len(r.records), "train_samples", r.sample_count())
print("val_records", len(v.records), "val_samples", v.sample_count())
PY
"$ROOT/.venv-holoboxing/bin/python" -c "import torch; print('GPU', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')"


### 5. 训练脚本（分块 + resume + 同步）

In [ ]:
%%writefile /content/sync_results.sh
#!/bin/bash
set -u
mkdir -p /content/drive/MyDrive/hbot/results /content/drive/MyDrive/hbot/logs
while true; do
  rsync -a /content/HoloMotion/results/ /content/drive/MyDrive/hbot/results/ 2>/dev/null || true
  rsync -a /content/logs/ /content/drive/MyDrive/hbot/logs/ 2>/dev/null || true
  sleep 300
done


In [ ]:
%%writefile /content/keepalive.py
import time
while True:
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "keepalive", flush=True)
    time.sleep(300)


In [ ]:
%%writefile /content/run_b7_sl.sh
#!/bin/bash
set -u
ROOT=/content/HoloMotion
cd "$ROOT" || exit 1
source /content/env-colab.sh
mkdir -p /content/logs /content/drive/MyDrive/hbot/results results/b7-sl
PY=$ROOT/.venv-holoboxing/bin/python
if pgrep -f "[p]ython -m holoboxing.training.train_sl" >/dev/null; then echo "SL already running"; exit 0; fi
if ls results/b7-sl*/epoch_000019.pt >/dev/null 2>&1; then
  cp "$(ls -t results/b7-sl*/epoch_000019.pt | head -1)" results/b7-sl/epoch_000019.pt
  echo "SL already done"; exit 0
fi
while true; do
  CKPT=$(ls -t results/b7-sl*/last.pt results/b7-sl*/epoch_*.pt 2>/dev/null | head -1 || true)
  OUT="results/b7-sl-c$(date +%Y%m%d_%H%M%S)"
  if [ -n "$CKPT" ]; then START="--resume $CKPT"; else START=""; fi
  LOG="/content/logs/b7sl_$(date +%Y%m%d_%H%M%S).log"
  nice -n 10 "$PY" -m holoboxing.training.train_sl \
    --assets "$HOLOBOXING_ASSETS" --lower "$HOLOBOXING_LOWER" \
    --data "$HOLOBOXING_DATA" --prior-checkpoint results/b1-prior/best.pt \
    --config holoboxing/training/configs/sample.json \
    --output "$OUT" --epochs 19 --batch-size 512 --seed 43 \
    --device cuda:0 --max-runtime-seconds 1800 $START 2>&1 \
    | tee "$LOG" | tee -a /content/drive/MyDrive/hbot/logs/b7sl_latest.log
  RC=${PIPESTATUS[0]}
  rsync -a results/ /content/drive/MyDrive/hbot/results/ 2>/dev/null || true
  if ls results/b7-sl*/epoch_000019.pt >/dev/null 2>&1; then
    cp "$(ls -t results/b7-sl*/epoch_000019.pt | head -1)" results/b7-sl/epoch_000019.pt
    echo "SL done"; break
  fi
  echo "SL chunk rc=$RC; sleep 10"; sleep 10
done


In [ ]:
%%writefile /content/run_b8.sh
#!/bin/bash
set -u
ROOT=/content/HoloMotion
cd "$ROOT" || exit 1
source /content/env-colab.sh
mkdir -p /content/logs results/b8-warmup results/b8-nfsp
PY=$ROOT/.venv-holoboxing/bin/python
SL=results/b7-sl/epoch_000019.pt
[ -f "$SL" ] || { echo "B7 SL not done"; exit 2; }
[ -f holoboxing/training/configs/b8.json ] || { echo "B8 code missing (b8-code.zip)"; exit 2; }
if pgrep -f "[p]ython -m holoboxing.training.train_rl" >/dev/null; then echo "RL already running"; exit 0; fi

# warmup 200 in 1800s chunks
while [ ! -f results/b8-warmup/model_200.pt ]; do
  if ls results/b8-warmup*/model_*.pt >/dev/null 2>&1; then
    CKPT=$(ls -t results/b8-warmup*/model_*.pt | head -1)
    START="--resume $CKPT"; OUT="results/b8-warmup-r$(date +%s)"
  else
    START="--supervised-checkpoint $SL"; OUT="results/b8-warmup"
  fi
  nice -n 10 "$PY" -m holoboxing.training.train_rl \
    --assets "$HOLOBOXING_ASSETS" --lower "$HOLOBOXING_LOWER" \
    --amp-motions "$HOLOBOXING_DATA" $START \
    --config holoboxing/training/configs/b8.json \
    --training-mode standing --output "$OUT" --num-envs 128 \
    --updates 200 --seed 43 --device cuda:0 --max-runtime-seconds 1800 \
    2>&1 | tee -a /content/logs/b8_warmup.log
  RC=${PIPESTATUS[0]}
  rsync -a results/ /content/drive/MyDrive/hbot/results/ 2>/dev/null || true
  echo "warmup chunk rc=$RC"; [ "$RC" -eq 0 ] || sleep 10
done

# NFSP 2000 in 200-update chunks
while true; do
  LATEST=$(ls -t results/b8-nfsp*/model_*.pt 2>/dev/null | head -1 || true)
  if [ -n "$LATEST" ]; then
    IT=$(basename "$LATEST" | sed -E 's/model_([0-9]+)\.pt/\1/')
    [ "$IT" -ge 2000 ] && break
    START="--resume $LATEST"; OUT="results/b8-nfsp-r$(date +%s)"
  else
    START="--policy-checkpoint results/b8-warmup/model_200.pt"; OUT="results/b8-nfsp"
  fi
  nice -n 10 "$PY" -m holoboxing.training.train_rl \
    --assets "$HOLOBOXING_ASSETS" --lower "$HOLOBOXING_LOWER" \
    --amp-motions "$HOLOBOXING_DATA" $START \
    --config holoboxing/training/configs/b8.json \
    --training-mode selfplay --opponent-pool-capacity 8 \
    --opponent-refresh-updates 32 --output "$OUT" --num-envs 128 \
    --updates 200 --seed 43 --device cuda:0 --max-runtime-seconds 1800 \
    2>&1 | tee -a /content/logs/b8_nfsp.log
  RC=${PIPESTATUS[0]}
  rsync -a results/ /content/drive/MyDrive/hbot/results/ 2>/dev/null || true
  echo "nfsp chunk rc=$RC"; [ "$RC" -eq 0 ] || sleep 10
done
echo B8_DONE


In [ ]:
#@title 6. 启动 B7 SL（tmux + nohup + nice）+ 监控
!chmod +x /content/run_b7_sl.sh /content/run_b8.sh /content/sync_results.sh
!tmux kill-server 2>/dev/null || true
!tmux new-session -d -s sync "nohup bash /content/sync_results.sh > /content/logs/sync.log 2>&1 &"
!tmux new-session -d -s b7sl "bash /content/run_b7_sl.sh > /content/logs/b7sl_stdout.log 2>&1"
!sleep 8
!tmux ls
!ps aux | grep -E "train_sl|run_b7_sl|sync_results" | grep -v grep
!tail -20 /content/logs/b7sl_stdout.log 2>/dev/null
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader


In [ ]:
#@title 7. 防 idle / 重连监视
# 7a. Colab 防闲置：每分钟模拟点击 Connect（保持浏览器标签页打开）
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
  console.log("keepalive");
  var b = document.querySelector("colab-connect-button");
  if (b) b.click();
}
setInterval(ClickConnect, 60000);
'''))

# 7b. 查看进度（重连后运行）
!tmux ls
!tmux capture-pane -pt b7sl 2>/dev/null | tail -30
!tail -30 /content/logs/b7sl_stdout.log 2>/dev/null
!tail -20 /content/drive/MyDrive/hbot/logs/b7sl_latest.log 2>/dev/null


In [ ]:
#@title 8. 启动 B8（需要 b8-code.zip；否则只提示）
!bash -c 'if [ -f /content/HoloMotion/holoboxing/training/configs/b8.json ]; then tmux kill-server 2>/dev/null || true; tmux new-session -d -s sync "nohup bash /content/sync_results.sh > /content/logs/sync.log 2>&1 &"; tmux new-session -d -s b8 "bash /content/run_b8.sh > /content/logs/b8_stdout.log 2>&1"; sleep 8; tmux ls; tail -20 /content/logs/b8_stdout.log; else echo "B8 code missing: upload b8-code.zip to Drive and rerun Cell 2"; fi'
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader


### 9. 导出 / validate / native

In [ ]:
%%bash
set -u
ROOT=/content/HoloMotion
cd "$ROOT" || exit 1
source /content/env-colab.sh
PY=$ROOT/.venv-holoboxing/bin/python
mkdir -p results
if [ -f results/b7-sl/epoch_000019.pt ]; then
  "$PY" -m holoboxing.training.export --checkpoint results/b7-sl/epoch_000019.pt --output results/boxing-b7-sl-only.onnx
  "$PY" -m holoboxing validate results/boxing-b7-sl-only.onnx \
    --assets "$HOLOBOXING_ASSETS" --lower "$HOLOBOXING_LOWER" \
    --output results/b7-sl-validation.json --benchmark || true
fi
# B8 avg/br export if B8 code + checkpoint exist
if [ -f holoboxing/training/configs/b8.json ]; then
  LATEST=$(ls -t results/b8-nfsp*/model_*.pt 2>/dev/null | head -1 || true)
  if [ -n "$LATEST" ]; then
    "$PY" -m holoboxing.training.export --checkpoint "$LATEST" --policy average --output results/boxing-b8-avg.onnx || true
    "$PY" -m holoboxing.training.export --checkpoint "$LATEST" --output results/boxing-b8-br.onnx || true
  fi
fi
rsync -a results/ /content/drive/MyDrive/hbot/results/ 2>/dev/null || true
ls -lh results/*.onnx 2>/dev/null


## 重连 / 续跑步骤

1. 重连 Colab 后，从 **Cell 1** 重新挂 Drive、**Cell 2** 重新解包数据/代码、**Cell 3** 恢复 venv（Drive 缓存）、**Cell 4** smoke。
2. 运行 **Cell 6** 启动 B7 SL：脚本会检测已完成的 `results/b7-sl/epoch_000019.pt`，有则直接退出；否则从最新 `last.pt/epoch_*.pt` 续跑。
3. B7 SL 完成后运行 **Cell 8** 启动 B8；同样按最新 `model_*.pt` 续跑，NFSP 每 200 updates 一块。
4. 每次运行 **Cell 7a** 的 JS keepalive，并保持浏览器标签页打开；**Cell 7b** 查看进度。
5. 所有 checkpoint/log 每 5 分钟 rsync 到 `/content/drive/MyDrive/hbot/`；Colab runtime 被回收也不会丢。

## 数据准备（不要在 Colab 跑）

在 ModelScope 实例上运行 `deployment/pack_colab_data.sh`，生成 `hbot-colab-data.tgz`，上传到私有 Drive `/MyDrive/hbot/`。包含：
`b7-boxing`（prepared 1,227 clip/741,842 samples，排除 robot_h5）、`models/holomotion-v2`、`holoboxing/assets`、`compiled`、`b1-prior`、`boxing-b1-sl.onnx`、`model_3000.pt`、`docs/b8`、`fall_attrib.py`。

## 注意

- `b8-code.zip` 目前尚未生成；没有它时 Cell 8 会提示并退出，B7 SL 仍可跑。
- T4 的 `validate` 时延不代表 RTX 5090；平台提交以官方验证为准。
- 免费 Colab 12h 上限；每块 1800s + checkpoint + resume。
- InterHuman/SMPL 只放私有 Drive/加密包，不公开分享。
